In [0]:
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.classification import GBTClassifier, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

import mlflow
import mlflow.spark
from mlflow.models.signature import infer_signature
import time

CATALOG = "fahrlytics"
GOLD    = f"{CATALOG}.gold"
MODEL_NAME = f"{CATALOG}.models.churn_classifier_gbt"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.models")

mlflow.set_experiment(f"/Users/{spark.sql('SELECT current_user()').collect()[0][0]}/fahrlytics_churn_prediction")

print(f"Catalog            : {CATALOG}")
print(f"Feature Source     : {GOLD}.customer_features")
print(f"Unity Catalog Target: {MODEL_NAME}")

In [0]:
# Create a dedicated volume for ML model checkpoints & staging
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.models.mlflow_staging")

In [0]:
print("Loading Gold customer features...")
raw_df = spark.table(f"{GOLD}.customer_features")

# Convert label to double for MLlib binary classification
df = raw_df.withColumn("label", F.col("target_is_churned").cast("double"))

total_records = df.count()
churn_count   = df.filter(F.col("label") == 1.0).count()
print(f"Total dataset records: {total_records:,}")
print(f"Churned users        : {churn_count:,} ({round(churn_count / total_records * 100, 2)}%)")

In [0]:
# Using an 80/20 train/test split with a fixed seed for reproducible benchmarks
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)

train_count = train_df.count()
test_count  = test_df.count()
print(f"Training set count : {train_count:,}")
print(f"Test set count     : {test_count:,}")

In [0]:
# Categorical columns to encode
categorical_cols = ["city", "device_type", "referral_source", "preferred_vehicle_type"]

# Numerical behavioral and anti-leakage velocity signals
numerical_cols = [
    "customer_tenure_days",
    "total_trips_requested",
    "total_trips_completed",
    "total_trips_cancelled",
    "total_spend_eur",
    "avg_fare_eur",
    "avg_distance_km",
    "avg_surge_multiplier",
    "rainy_day_trip_ratio",
    "weekend_trip_ratio",
    "cancellation_rate",
    "trips_last_30d",
    "trips_prior_60d",
    "trip_velocity_ratio",
    "total_payment_attempts",
    "failed_payment_count",
    "fare_dispute_count",
    "payment_failure_rate",
    "paypal_preference_ratio",
    "total_app_sessions",
    "total_ride_searches",
    "total_confirmed_sessions",
    "total_bounced_sessions",
    "search_to_booking_conversion",
    "app_bounce_rate",
    "avg_session_duration_sec"
]

stages = []

# Index and One-Hot Encode categorical attributes
for col_name in categorical_cols:
    indexer = StringIndexer(inputCol=col_name, outputCol=f"{col_name}_idx", handleInvalid="keep")
    encoder = OneHotEncoder(inputCols=[f"{col_name}_idx"], outputCols=[f"{col_name}_vec"])
    stages += [indexer, encoder]

# Assemble numeric and categorical vectors
assembler_inputs = [f"{c}_vec" for c in categorical_cols] + numerical_cols
assembler = VectorAssembler(inputCols=assembler_inputs, outputCol="raw_features", handleInvalid="keep")
scaler    = StandardScaler(inputCol="raw_features", outputCol="features", withStd=True, withMean=False)

stages += [assembler, scaler]

# Estimator: Gradient-Boosted Decision Trees
gbt = GBTClassifier(
    featuresCol="features",
    labelCol="label",
    maxIter=30,
    maxDepth=5,
    stepSize=0.1,
    seed=42
)
stages.append(gbt)

pipeline = Pipeline(stages=stages)

In [0]:
UC_VOLUME_TMP = f"/Volumes/{CATALOG}/models/mlflow_staging"

mlflow.spark.log_model(
    spark_model=model,
    artifact_path="churn_model_pipeline",
    signature=signature,
    dfs_tmpdir=UC_VOLUME_TMP   
)

In [0]:
evaluator_roc = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
evaluator_pr  = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderPR")
evaluator_acc = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
evaluator_f1  = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")

print("Initiating MLflow run for Churn Prediction...")

mlflow.end_run()

with mlflow.start_run(run_name="gbt_customer_churn_baseline") as run:
    run_id = run.info.run_id
    t0 = time.time()
    
    # Fit the pipeline on training data
    model = pipeline.fit(train_df)
    train_duration = time.time() - t0
    
    # Generate predictions on test data
    predictions = model.transform(test_df)
    
    # Evaluate performance
    auc_roc = evaluator_roc.evaluate(predictions)
    auc_pr  = evaluator_pr.evaluate(predictions)
    acc     = evaluator_acc.evaluate(predictions)
    f1      = evaluator_f1.evaluate(predictions)
    
    # Log parameters
    mlflow.log_param("model_type", "GBTClassifier")
    mlflow.log_param("max_iter", 30)
    mlflow.log_param("max_depth", 5)
    mlflow.log_param("step_size", 0.1)
    mlflow.log_param("train_samples", train_count)
    mlflow.log_param("test_samples", test_count)
    
    # Log evaluation metrics
    mlflow.log_metric("test_area_under_roc", auc_roc)
    mlflow.log_metric("test_area_under_pr", auc_pr)
    mlflow.log_metric("test_accuracy", acc)
    mlflow.log_metric("test_f1_score", f1)
    mlflow.log_metric("training_duration_sec", train_duration)
    
    print("\n--- MODEL PERFORMANCE RESULTS ---")
    print(f"  Test ROC-AUC  : {auc_roc:.4f}")
    print(f"  Test PR-AUC   : {auc_pr:.4f}")
    print(f"  Test Accuracy : {acc * 100:.2f}%")
    print(f"  Test F1-Score : {f1:.4f}")
    print(f"  Training Time : {train_duration:.2f}s")
    
    # Extract feature importance from fitted GBT stage
    gbt_stage = model.stages[-1]
    importances = gbt_stage.featureImportances.toArray()
    
    # Log model artifacts to MLflow
    sample_input = train_df.select(categorical_cols + numerical_cols).limit(5).toPandas()
    sample_pred  = predictions.select("prediction").limit(5).toPandas()
    signature    = infer_signature(sample_input, sample_pred)
    
    mlflow.spark.log_model(
        spark_model=model,
        artifact_path="churn_model_pipeline",
        signature=signature,
        dfs_tmpdir=UC_VOLUME_TMP
    )
    print(f"\nMLflow Model logged successfully with Run ID: {run_id}")

In [0]:
print("\nTop 10 Drivers of Customer Churn:")
feature_names = assembler_inputs
feature_importance_tuples = sorted(zip(assembler_inputs, importances), key=lambda x: x[1], reverse=True)

for rank, (feat, score) in enumerate(feature_importance_tuples[:10], start=1):
    print(f"  {rank:>2}. {feat:<30} : {score * 100:.2f}%")

In [0]:
print(f"\nRegistering model to Unity Catalog: {MODEL_NAME}...")

model_uri = f"runs:/{run_id}/churn_model_pipeline"
registered_model = mlflow.register_model(
    model_uri=model_uri,
    name=MODEL_NAME
)

print(f"Model registered as Version {registered_model.version} in Unity Catalog.")

In [0]:
print("Scoring customer base to output top retention priority leads...")

scored_df = model.transform(df)

high_risk_retention_list = (
    scored_df.filter((F.col("prediction") == 1.0) & (F.col("total_spend_eur") >= 100.0))
    .select(
        "customer_id",
        "city",
        "total_spend_eur",
        "trip_velocity_ratio",
        "trips_last_30d",
        "payment_failure_rate",
        "prediction"
    )
    .orderBy(F.col("total_spend_eur").desc())
    .limit(10)
)

print("\nTop High-Value At-Risk Customers (Ready for CRM Activation):")
high_risk_retention_list.show(truncate=False)